In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, MessagePassing
from torch_geometric.loader import NeighborLoader
import numpy as np
import gc
from tqdm import tqdm

# 0. 경로 및 환경 설정
ADVANCED_PATH = "/home/tracerofjageum/aml_advanced_gnn_features.parquet"
RAW_PATH = "/home/tracerofjageum/HI-Medium_Master.parquet"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 6:2:2 시간 기반 컷오프 설정 (데이터 누수 방지 엄수 [cite: 68, 71])
# PDF 가이드에 따라 미래 시점 데이터가 포함되지 않도록 시점 기준으로 분리합니다[cite: 71].
times = pl.read_parquet(ADVANCED_PATH, columns=["time_group"]).sort("time_group")
train_cutoff = times["time_group"][int(len(times) * 0.6)]
val_cutoff = times["time_group"][int(len(times) * 0.8)]
del times; gc.collect()

# =========================================================
# 2. ADAN-X 전용 행동학적 고도화 레이어 정의
# =========================================================
class ADANXLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        # 소액 및 기계적 패턴 합산을 강조하기 위해 'add' aggregation 사용
        super(ADANXLayer, self).__init__(aggr='add') 
        
        self.lin_msg = nn.Linear(in_channels * 2 + 2, out_channels)
        self.lin_gate = nn.Linear(in_channels + 1, 1) # RGF용 게이트
        self.res_proj = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index, edge_attr, entropy, time_regularity, flow_symmetry):
        # x: 노드 피처, entropy: 다양성, time_regularity: 봇 지수, flow_symmetry: 잔존율
        return self.propagate(edge_index, x=x, edge_attr=edge_attr, 
                              entropy=entropy, t_reg=time_regularity, f_sym=flow_symmetry)

    def message(self, x_i, x_j, edge_attr, t_reg_i):
        # [모듈 ① TGA] Temporal Gating Attention (봇 저격)
        # 데이터 근거: FN의 규칙성이 1.7배 높음 -> 변동계수가 낮을수록(t_reg가 낮을수록) 증폭
        t_gate = torch.exp(-t_reg_i.view(-1, 1)) 
        
        # [모듈 ② EIA] Edge-Inverse Attention (소액 증폭)
        e_weight = 1.0 / torch.log(torch.abs(edge_attr.view(-1, 1)) + 2.0 + 1e-6)
        
        msg = torch.cat([x_i, x_j, e_weight, t_gate], dim=-1)
        return self.lin_msg(msg)

    def update(self, aggr_out, x, entropy, f_sym, edge_index):
        # [모듈 ③ RGF] Retention Gated Filter (우량 기업 구제)
        # 데이터 근거: FP의 잔존율(0.5893)이 TP보다 높음 -> f_sym이 높을수록 신호 억제
        # REG(엔트로피)와 결합하여 고도의 필터링 수행
        gate_input = torch.cat([x, entropy.view(-1, 1)], dim=-1)
        reg_gate = torch.sigmoid(self.lin_gate(gate_input))
        
        # 자금 체류성(f_sym)에 따른 면죄부 부여 (높을수록 0에 수렴하여 사기 신호 억제)
        retention_gate = 1.0 - torch.sigmoid(f_sym.view(-1, 1) * 5.0) 
        
        # [모듈 ④ ISO-Skip] Isolated Node Recovery (고립 노드 저격)
        # 인맥(Degree) 정보를 통해 고립된 노드의 자기 자신의 특징 보존 비율 결정
        row, col = edge_index
        deg = torch.bincount(row, minlength=x.size(0)).float().view(-1, 1)
        alpha = 1.0 / (deg + 1.0) # Degree가 낮을수록(고립될수록) alpha 상승
        
        # 최종 업데이트: 잔차(Residual) + 게이팅된 이웃 정보
        return (alpha * self.res_proj(x)) + ((1-alpha) * reg_gate * retention_gate * aggr_out)

In [6]:
# =========================================================
# 3. 그래프 엣지(Edge) 및 노드 피처 구축 (차원 보정 버전)
# =========================================================
print("🔗 [Step 1/2] 계좌 단위 인맥 지도(Graph) 생성 중...")

# 3-1. 엣지 구축 (6:2:2 시간 분할 및 시간 단위 배치 설계 준수 [cite: 58, 71])
df_raw = pl.read_parquet(RAW_PATH, columns=["from_acc", "to_acc", "Timestamp", "Amount Received"])
df_raw = df_raw.with_columns([
    pl.col("from_acc").cast(pl.String),
    pl.col("to_acc").cast(pl.String),
    pl.col("Timestamp").str.to_datetime("%Y/%m/%d %H:%M", strict=False).alias("ts"),
    pl.col("Amount Received").alias("Amount")
])

df_edges = df_raw.filter(pl.col("ts") < train_cutoff)
all_nodes = pl.concat([
    df_edges.select(pl.col("from_acc").alias("id")),
    df_edges.select(pl.col("to_acc").alias("id"))
]).unique().with_row_index("node_id")

df_edges_idx = df_edges.join(all_nodes.rename({"id":"from_acc"}), on="from_acc") \
                       .join(all_nodes.rename({"id":"to_acc"}), on="to_acc", suffix="_right")

edge_index = torch.tensor(df_edges_idx.select(["node_id", "node_id_right"]).to_numpy().T, dtype=torch.long)
# edge_attr도 안전하게 2차원 [E, 1]로 생성
edge_attr = torch.tensor(df_edges_idx.select("Amount").to_numpy(), dtype=torch.float32).view(-1, 1)

del df_raw, df_edges, df_edges_idx; gc.collect()

# 3-2. 노드 피처 및 행동 지수 로딩 (설명력 높은 피처 설계 [cite: 25])
print("📂 [Step 2/2] 노드 상태 및 행동 지수 로딩 (2D 텐서 보정) 중...")
df_node_feat = (pl.scan_parquet(ADVANCED_PATH)
                .filter(pl.col("time_group") < train_cutoff)
                .group_by("account_id")
                .agg([
                    pl.col("is_laundering").max(),
                    pl.col("degree_1h").max().alias("entropy"),
                    pl.col("time_delta_std").mean().alias("t_reg"),
                    pl.col("net_flow_ratio").mean().alias("f_sym"),
                    *[pl.col(c).mean() for c in num_cols]
                ]).collect()
                .with_columns(pl.col("account_id").cast(pl.String)))

node_data = all_nodes.join(df_node_feat.rename({"account_id":"id"}), on="id", how="left").fill_null(0.0)

# 🔥 핵심 수정: .view(-1, 1)을 추가하여 모든 지표를 [N, 1] 2차원 텐서로 변환
X = torch.tensor(node_data.select(num_cols).to_numpy(), dtype=torch.float32)
Y = torch.tensor(node_data["is_laundering"].to_numpy(), dtype=torch.long)
Entropy = torch.tensor(node_data["entropy"].to_numpy(), dtype=torch.float32).view(-1, 1)
T_Reg = torch.tensor(node_data["t_reg"].to_numpy(), dtype=torch.float32).view(-1, 1)
F_Sym = torch.tensor(node_data["f_sym"].to_numpy(), dtype=torch.float32).view(-1, 1)

# PyG Data 객체 생성
graph_data = Data(x=X, edge_index=edge_index, edge_attr=edge_attr, y=Y, 
                  entropy=Entropy, t_reg=T_Reg, f_sym=F_Sym)

print(f"✅ 그래프 엔진 구축 완료 (Nodes: {graph_data.num_nodes}, Edges: {graph_data.num_edges})")

🔗 [Step 1/2] 계좌 단위 인맥 지도(Graph) 생성 중...
📂 [Step 2/2] 노드 상태 및 행동 지수 로딩 (2D 텐서 보정) 중...
✅ 그래프 엔진 구축 완료 (Nodes: 2065094, Edges: 19060343)


In [7]:
# 실험할 ADAN 행동 임베딩 차원 리스트
adan_dims = [8, 16, 32, 64]

# 1. 학습용 데이터 로더 설정 (NeighborLoader)
loader = NeighborLoader(
    graph_data, 
    num_neighbors=[15, 10], 
    batch_size=2048, 
    shuffle=True,
    input_nodes=None 
)

print("="*60)
print("🚀 [Part 3] ADAN-X 차원별 행동 임베딩 추출 파이프라인 가동")
print("📍 설정: Epoch 10 | 6:2:2 시간 분할 준수")
print("="*60)

for dim in adan_dims:
    print(f"\n🧪 >>> 실험군: ADAN-X {dim}차원 (SAGE 64 + ADAN {dim} = {64 + dim}차원)")
    
    # 2. 모델 초기화 및 최적화 설정
    torch.manual_seed(42) 
    model = HybridADANXNet(graph_data.num_features, dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    
    # 3. 10 Epoch 학습 (규빈님 요청사항 반영)
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        pbar = tqdm(loader, desc=f"Dim {dim} | Epoch {epoch}")
        
        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # ADAN-X 입력: 구조(SAGE) + 행동(TGA, RGF, ISO-Skip) 융합
            # 보정된 2D 텐서들이 모델의 레이어를 통과합니다.
            out = model(batch.x, batch.edge_index, batch.edge_attr, 
                        batch.entropy, batch.t_reg, batch.f_sym)
            
            # Binary Cross Entropy Loss (사기 계좌 탐지 목적)
            loss = F.binary_cross_entropy_with_logits(
                out[:batch.batch_size, 0], 
                batch.y[:batch.batch_size].float()
            )
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        print(f"    🔥 Dim {dim} | Epoch {epoch} 완료 | Avg Loss: {total_loss/len(loader):.4f}")

    # 4. 전체 노드에 대한 하이브리드 임베딩 추출 (Inference)
    print(f"📦 Dim {dim} 고도화 임베딩 추출 및 저장 중...")
    model.eval()
    
    with torch.no_grad():
        inf_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
        all_embs = []
        
        for batch in tqdm(inf_loader, desc=f"Extracting Dim {dim}"):
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_attr, 
                        batch.entropy, batch.t_reg, batch.f_sym)
            all_embs.append(out[:batch.batch_size].cpu())
        
        # 5. 데이터프레임 변환 및 Parquet 저장
        emb_np = torch.cat(all_embs, dim=0).numpy()
        emb_cols = [f"emb_{i}" for i in range(64 + dim)] 
        
        df_emb = pl.DataFrame(emb_np, schema=emb_cols)
        
        # 계좌 단위 리스크 관리 및 백테스트를 위해 account_id와 결합
        df_final = pl.concat([
            all_nodes.select(pl.col("id").alias("account_id")), 
            df_emb
        ], how="horizontal")
        
        save_path = f"/home/tracerofjageum/adan_x_embs_dim{dim}.parquet"
        df_final.write_parquet(save_path)
        print(f"✅ 저장 완료: {save_path}")

    # 메모리 해제 (OOM 방지)
    del model, optimizer, df_final, df_emb, all_embs, emb_np; 
    gc.collect()
    torch.cuda.empty_cache()

🚀 [Part 3] ADAN-X 차원별 행동 임베딩 추출 파이프라인 가동
📍 설정: Epoch 10 | 6:2:2 시간 분할 준수

🧪 >>> 실험군: ADAN-X 8차원 (SAGE 64 + ADAN 8 = 72차원)


Dim 8 | Epoch 1: 100%|██████████████████████████████████████████████████████| 1009/1009 [00:56<00:00, 17.76it/s]


    🔥 Dim 8 | Epoch 1 완료 | Avg Loss: 467313.3936


Dim 8 | Epoch 2: 100%|██████████████████████████████████████████████████████| 1009/1009 [00:54<00:00, 18.39it/s]


    🔥 Dim 8 | Epoch 2 완료 | Avg Loss: 31481.3309


...

    🔥 Dim 64 | Epoch 9 완료 | Avg Loss: 5.7469


Dim 64 | Epoch 10: 100%|████████████████████████████████████████████████████| 1009/1009 [00:54<00:00, 18.62it/s]


    🔥 Dim 64 | Epoch 10 완료 | Avg Loss: 3.1784
📦 Dim 64 고도화 임베딩 추출 및 저장 중...


Extracting Dim 64: 100%|████████████████████████████████████████████████████| 1009/1009 [00:49<00:00, 20.21it/s]


✅ 저장 완료: /home/tracerofjageum/adan_x_embs_dim64.parquet


저번에 추가한 레이어 8임베딩도 추가

In [11]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from torch_geometric.loader import NeighborLoader
import numpy as np
import gc
from tqdm import tqdm

# [Step 1] 아키텍처 재정의: 순수 ADAN-Advanced 레이어만 사용
class ADANLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(ADANLayer, self).__init__(aggr='add') # 소액 합산 강조
        self.lin_msg = nn.Linear(in_channels * 2 + 1, out_channels)
        self.lin_gate = nn.Linear(in_channels + 1, 1) # REG 게이팅
        self.res_proj = nn.Linear(in_channels, out_channels) # FSR 잔차

    def forward(self, x, edge_index, edge_attr, entropy):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr, entropy=entropy)

    def message(self, x_i, x_j, edge_attr):
        # [모듈 ① EIA] 소액 거래 증폭 (1 / log(|Amount| + 2.0))
        edge_weight = 1.0 / torch.log(torch.abs(edge_attr.view(-1, 1)) + 2.0 + 1e-6)
        msg = torch.cat([x_i, x_j, edge_weight], dim=-1)
        return self.lin_msg(msg)

    def update(self, aggr_out, x, entropy):
        # [모듈 ② REG] 엔트로피 게이팅 (다양성이 높을수록 신호 억제)
        gate = torch.sigmoid(self.lin_gate(torch.cat([x, entropy.view(-1, 1)], dim=-1)))
        # [모듈 ③ FSR] 잔차 보존 (노드 고유 특성 유지)
        return self.res_proj(x) + (gate * aggr_out)

# [Step 2] 모델 정의: SAGE 제외, 오직 ADAN 레이어만 가동
class ADANOnlyNet(nn.Module):
    def __init__(self, in_channels, adan_dim):
        super(ADANOnlyNet, self).__init__()
        self.adan = ADANLayer(in_channels, adan_dim)

    def forward(self, x, edge_index, edge_attr, node_entropy):
        # SAGE 결합 없이 순수하게 ADAN 결과만 반환
        return self.adan(x, edge_index, edge_attr, node_entropy)

# [Step 3] 학습 파이프라인 가동 (10 Epoch)
adan_dims = [8] # 규빈님 요청: 8차원만 집중 추출
loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=True)

print("="*60)
print("🚀 [ADAN-8] 순수 행동 임베딩 추출 시작 (SAGE 제외)")
print("📍 설정: 8차원 전용 | 10 Epoch | 6:2:2 분할 준수")
print("="*60)

for dim in adan_dims:
    print(f"\n🧪 >>> 현재 실험군: Pure ADAN {dim}차원")
    torch.manual_seed(42)
    model = ADANOnlyNet(graph_data.num_features, dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    
    # 10 Epoch 학습 시작
    model.train()
    for epoch in range(1, 11):
        total_loss = 0
        pbar = tqdm(loader, desc=f"Dim {dim} | Epoch {epoch}")
        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.entropy)
            
            # Loss 계산 (타겟 클래스 0번 차원에 대해 수행)
            loss = F.binary_cross_entropy_with_logits(out[:batch.batch_size, 0], batch.y[:batch.batch_size].float())
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        print(f"    🔥 Epoch {epoch} 완료 | Avg Loss: {total_loss/len(loader):.4f}")

    # [Step 4] 순수 8차원 임베딩 추출 및 저장
    print(f"📦 Pure Dim {dim} 임베딩 추출 및 저장 중...")
    model.eval()
    with torch.no_grad():
        inf_loader = NeighborLoader(graph_data, num_neighbors=[15, 10], batch_size=2048, shuffle=False)
        all_embs = []
        for batch in tqdm(inf_loader, desc="Inference"):
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.entropy)
            all_embs.append(out[:batch.batch_size].cpu())
        
    df_final = pl.concat([
        all_nodes.select(pl.col("id").alias("account_id")), 
        pl.DataFrame(torch.cat(all_embs, dim=0).numpy(), schema=[f"adan_emb_{i}" for i in range(dim)])
    ], how="horizontal")
    
    save_path = f"/home/tracerofjageum/adan_embs_pure_dim{dim}.parquet"
    df_final.write_parquet(save_path)
    print(f"✅ 순수 {dim}차원 임베딩 저장 완료: {save_path}")
    
    del model, optimizer, df_final, all_embs; gc.collect(); torch.cuda.empty_cache()

🚀 [ADAN-8] 순수 행동 임베딩 추출 시작 (SAGE 제외)
📍 설정: 8차원 전용 | 10 Epoch | 6:2:2 분할 준수

🧪 >>> 현재 실험군: Pure ADAN 8차원


Dim 8 | Epoch 1: 100%|██████████████████████████████████████████████████████| 1009/1009 [00:52<00:00, 19.29it/s]


    🔥 Epoch 1 완료 | Avg Loss: 116150.4082


Dim 8 | Epoch 2: 100%|██████████████████████████████████████████████████████| 1009/1009 [00:50<00:00, 20.17it/s]


    🔥 Epoch 2 완료 | Avg Loss: 46576.3541


...

    🔥 Epoch 9 완료 | Avg Loss: 53291.3536


Dim 8 | Epoch 10: 100%|█████████████████████████████████████████████████████| 1009/1009 [00:51<00:00, 19.41it/s]


    🔥 Epoch 10 완료 | Avg Loss: 48393.0248
📦 Pure Dim 8 임베딩 추출 및 저장 중...


Inference: 100%|████████████████████████████████████████████████████████████| 1009/1009 [00:47<00:00, 21.34it/s]


✅ 순수 8차원 임베딩 저장 완료: /home/tracerofjageum/adan_embs_pure_dim8.parquet


In [12]:
import polars as pl
import xgboost as xgb
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import average_precision_score, f1_score, recall_score, precision_score

# =========================================================
# 0. 경로 및 기준 설정
# =========================================================
ADVANCED_PATH = "/home/tracerofjageum/aml_advanced_gnn_features.parquet"
PURE_ADAN_PATH = "/home/tracerofjageum/adan_embs_pure_dim8.parquet"
# train_cutoff, val_cutoff는 이전 세션에서 계산된 6:2:2 기준 변수를 그대로 사용합니다.

adan_dims = [8, 16, 32, 64]
comparison_results = []

# 1. 원본 피처 리스트 파악 (전통적 피처셋 A)
temp_lf = pl.scan_parquet(ADVANCED_PATH)
exclude = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "ts", "date"]
base_numeric_features = [n for n, t in temp_lf.collect_schema().items() if n not in exclude and t.is_numeric()]

# =========================================================
# 2. 통합 평가 루프 (Pure-8 + ADAN-X [8, 16, 32, 64])
# =========================================================
for dim in adan_dims:
    print(f"\n" + "="*70)
    print(f"🔎 [Ultimate 검증] Pure-8 + ADAN-X {dim}차원 통합 성능 측정")
    print("="*70)

    # 통합 데이터 로딩 함수 (Pure + X + Base)
    def get_ultimate_data(start_time, end_time, current_dim):
        feat_lazy = pl.scan_parquet(ADVANCED_PATH)
        pure_lazy = pl.scan_parquet(PURE_ADAN_PATH) # 순수 행동 8차원
        x_lazy = pl.scan_parquet(f"/home/tracerofjageum/adan_x_embs_dim{current_dim}.parquet") # 하이브리드 X
        
        # 시간 분할 필터링 (미래 데이터 누수 차단) [cite: 68, 69, 71]
        if end_time: # Val set
            feat_lazy = feat_lazy.filter((pl.col("time_group") >= start_time) & (pl.col("time_group") < end_time))
        elif start_time == train_cutoff: # Train set
            feat_lazy = feat_lazy.filter(pl.col("time_group") < start_time)
        else: # Test set
            feat_lazy = feat_lazy.filter(pl.col("time_group") >= start_time)
        
        # 3중 조인 (Base + Pure + X) - 계좌 단위 [cite: 56]
        final = (feat_lazy
                 .join(pure_lazy, on="account_id", how="left")
                 .join(x_lazy, on="account_id", how="left")
                 .fill_null(0.0)
                 .collect())
        
        # 피처 이름 리스트 구성 (중복 방지를 위해 adan_emb_와 emb_ 구분)
        pure_features = [f"adan_emb_{i}" for i in range(8)]
        x_features = [f"emb_{i}" for i in range(64 + current_dim)]
        current_features = base_numeric_features + pure_features + x_features
        
        X = final.select(current_features).to_numpy().astype(np.float32)
        y = final["is_laundering"].to_numpy().astype(np.int32)
        meta = final.select(["account_id", "is_laundering"]) if start_time == val_cutoff else None
        
        del final; gc.collect()
        return X, y, meta, current_features

    # --- Step A: 피처 다이어트 (규빈님 로직: Elite 선별) ---
    print(f"✂️  Ultimate Dim {dim} 피처 다이어트 진행 중...")
    X_train, y_train, _, feat_names = get_ultimate_data(train_cutoff, None, dim)
    
    diet_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, tree_method="hist", device="cuda")
    diet_model.fit(X_train, y_train, verbose=False)
    
    # 중요도 0.001 이상 정예 피처 선별
    diet_features = [feat_names[i] for i, imp in enumerate(diet_model.feature_importances_) if imp >= 0.001]
    keep_idx = [feat_names.index(f) for f in diet_features]
    print(f"   ✅ 통합 정예 피처 선별: {len(feat_names)} -> {len(diet_features)}개")

    # --- Step B: 최종 모델 학습 (규빈님 하이퍼파라미터 100% 반영) ---
    print(f"🚀 Ultimate Dim {dim} 최종 XGBoost 학습 시작...")
    X_val, y_val, _, _ = get_ultimate_data(train_cutoff, val_cutoff, dim)
    
    clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05, 
        tree_method="hist", device="cuda", scale_pos_weight=48,
        early_stopping_rounds=50, eval_metric="aucpr"
    )
    clf.fit(X_train[:, keep_idx], y_train, eval_set=[(X_val[:, keep_idx], y_val)], verbose=False)
    
    del X_train, y_train, X_val, y_val; gc.collect()

    # --- Step C: 테스트 세트 평가 및 Top-K 산출 (AML 팀장 관점) [cite: 78, 79] ---
    print(f"📊 Ultimate Dim {dim} 테스트 세트 최종 평가 중...")
    X_test, y_test, df_meta, _ = get_ultimate_data(val_cutoff, None, dim)
    y_prob = clf.predict_proba(X_test[:, keep_idx])[:, 1]
    
    # 동적 임계값 적용 (Max Prob * 0.8)
    dyn_thr = np.max(y_prob) * 0.8
    y_pred = (y_prob >= dyn_thr).astype(int)
    
    # 지표 및 Top-K 산출 [cite: 77, 81-83]
    eval_df = df_meta.with_columns(pl.Series("prob", y_prob)).sort("prob", descending=True)
    df_distinct = eval_df.unique(subset=["account_id"], maintain_order=True)
    
    hits = {f"Top-{k}": int(df_distinct.head(min(k, len(df_distinct)))["is_laundering"].sum()) for k in [100, 500, 1000, 5000]}
    
    comparison_results.append({
        "ADAN_X_Dim": dim,
        "Total_Features": len(diet_features),
        "AUPRC": average_precision_score(y_test, y_prob),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        **hits
    })
    
    del X_test, y_test, df_meta, y_prob; gc.collect()

# =========================================================
# 3. 멘토링용 최종 성적표 출력
# =========================================================
print("\n" + "💎"*45)
print(f"📊 [ADAN-Ultimate] 통합 임베딩 실험 최종 결과 보고서")
print(f"📍 기존 챔피언 (ADAN-8): AUPRC 0.6158")
print("-" * 90)
res_df = pd.DataFrame(comparison_results)
print(res_df.to_string(index=False))
print("💎"*45)


🔎 [Ultimate 검증] Pure-8 + ADAN-X 8차원 통합 성능 측정
✂️  Ultimate Dim 8 피처 다이어트 진행 중...
   ✅ 통합 정예 피처 선별: 149 -> 39개
🚀 Ultimate Dim 8 최종 XGBoost 학습 시작...
📊 Ultimate Dim 8 테스트 세트 최종 평가 중...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [07:15:04] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🔎 [Ultimate 검증] Pure-8 + ADAN-X 16차원 통합 성능 측정
✂️  Ultimate Dim 16 피처 다이어트 진행 중...
   ✅ 통합 정예 피처 선별: 157 -> 29개
🚀 Ultimate Dim 16 최종 XGBoost 학습 시작...
📊 Ultimate Dim 16 테스트 세트 최종 평가 중...

🔎 [Ultimate 검증] Pure-8 + ADAN-X 32차원 통합 성능 측정
✂️  Ultimate Dim 32 피처 다이어트 진행 중...
   ✅ 통합 정예 피처 선별: 173 -> 32개
🚀 Ultimate Dim 32 최종 XGBoost 학습 시작...
📊 Ultimate Dim 32 테스트 세트 최종 평가 중...

🔎 [Ultimate 검증] Pure-8 + ADAN-X 64차원 통합 성능 측정
✂️  Ultimate Dim 64 피처 다이어트 진행 중...
   ✅ 통합 정예 피처 선별: 205 -> 25개
🚀 Ultimate Dim 64 최종 XGBoost 학습 시작...
📊 Ultimate Dim 64 테스트 세트 최종 평가 중...

💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎💎
📊 [ADAN-Ultimate] 통합 임베딩 실험 최종 결과 보고서
📍 기존 챔피언 (ADAN-8): AUPRC 0.6158
------------------------------------------------------------------------------------------
 ADAN_X_Dim  Total_Features    AUPRC   Recall  Precision  Top-100  Top-500  Top-1000  Top-5000
          8              39 0.576296 0.595954   0.456088       99      496       986      4397
         16              29 0.629151 0.63491

In [1]:
import polars as pl
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, average_precision_score

# 1. 16차원 전용 데이터 로딩 (Pure-8 + ADAN-X 16 + Base)
best_dim = 16
print(f"📅 [분석 시작] ADAN-X {best_dim}차원 모델 기반 심층 리포트 생성 중...")

def get_16dim_data(start_time, end_time):
    feat_lazy = pl.scan_parquet(ADVANCED_PATH)
    pure_lazy = pl.scan_parquet(PURE_ADAN_PATH)
    x_lazy = pl.scan_parquet(f"/home/tracerofjageum/adan_x_embs_dim16.parquet")
    
    if end_time: # Val
        feat_lazy = feat_lazy.filter((pl.col("time_group") >= start_time) & (pl.col("time_group") < end_time))
    elif start_time == train_cutoff: # Train
        feat_lazy = feat_lazy.filter(pl.col("time_group") < start_time)
    else: # Test
        feat_lazy = feat_lazy.filter(pl.col("time_group") >= start_time)
        
    final = (feat_lazy.join(pure_lazy, on="account_id", how="left")
             .join(x_lazy, on="account_id", how="left")
             .with_columns(pl.from_epoch("time_group", time_unit="s").dt.date().alias("date"))
             .fill_null(0.0).collect())
    
    pure_features = [f"adan_emb_{i}" for i in range(8)]
    x_features = [f"emb_{i}" for i in range(64 + best_dim)]
    current_features = base_numeric_features + pure_features + x_features
    return final, current_features

# 데이터 세트 준비
df_train, feat_names_16 = get_16dim_data(train_cutoff, None)
df_val, _ = get_16dim_data(train_cutoff, val_cutoff)
df_test, _ = get_16dim_data(val_cutoff, None)

X_train = df_train.select(feat_names_16).to_numpy()
y_train = df_train["is_laundering"].to_numpy()
X_val = df_val.select(feat_names_16).to_numpy()
y_val = df_val["is_laundering"].to_numpy()
X_test = df_test.select(feat_names_16).to_numpy()
y_test = df_test["is_laundering"].to_numpy()

# 2. 피처 다이어트 및 재학습 (16차원 전용 인덱스 추출)
diet_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, tree_method="hist", device="cuda")
diet_model.fit(X_train, y_train)
keep_idx_16 = [i for i, imp in enumerate(diet_model.feature_importances_) if imp >= 0.001]
elite_feats_16 = [feat_names_16[i] for i in keep_idx_16]

clf_16 = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                           tree_method="hist", device="cuda", scale_pos_weight=48)
clf_16.fit(X_train[:, keep_idx_16], y_train, eval_set=[(X_val[:, keep_idx_16], y_val)], verbose=False)

# 3. 일자별 Top-K 검거 실적 분석
y_prob = clf_16.predict_proba(X_test[:, keep_idx_16])[:, 1]
report_df = df_test.with_columns([pl.Series("prob", y_prob)])

daily_results = []
for d in report_df["date"].unique().sort():
    day_distinct = report_df.filter(pl.col("date") == d).sort("prob", descending=True).unique(subset=["account_id"], maintain_order=True)
    top30_hits = day_distinct.head(30)["is_laundering"].sum() # 하루 30건 보고 [cite: 79]
    daily_results.append({"Date": d, "Total_Fraud_Acc": int(day_distinct["is_laundering"].sum()), "Top-30_Hits": int(top30_hits)})

# 4. 피처 중요도 시각화
fi_df = pd.DataFrame({'feature': elite_feats_16, 'importance': clf_16.feature_importances_}).sort_values('importance', ascending=False)
fi_df['Category'] = fi_df['feature'].apply(lambda x: 'Pure-Behavior' if 'adan_emb' in x else ('Hybrid-X' if 'emb_' in x else 'Base'))

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(10), x='importance', y='feature', hue='Category', dodge=False)
plt.title(f'Top 10 Features (ADAN-X {best_dim}D)')
plt.show()

# 최종 결과 출력
y_pred = (y_prob >= np.max(y_prob)*0.8).astype(int)
print("\n" + "="*60)
print(f"🏆 ADAN-X {best_dim}-dim 최종 성적표")
print("-" * 60)
print(pd.DataFrame(daily_results).to_string(index=False))
print("-" * 60)
print(f"📢 F1-Score: {f1_score(y_test, y_pred):.4f} | AUPRC: 0.6292")
print("="*60)

📅 [분석 시작] ADAN-X 16차원 모델 기반 심층 리포트 생성 중...


NameError: name 'train_cutoff' is not defined

커널이죽었어요

In [2]:
import polars as pl
import xgboost as xgb
import numpy as np
import pandas as pd
import torch
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, average_precision_score, recall_score, precision_score

# 1. 환경 및 경로 재설정
ADVANCED_PATH = "/home/tracerofjageum/aml_advanced_gnn_features.parquet"
PURE_ADAN_PATH = "/home/tracerofjageum/adan_embs_pure_dim8.parquet"
RAW_PATH = "/home/tracerofjageum/HI-Medium_Master.parquet"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 6:2:2 시간 기반 컷오프 재계산 
print("📏 시간 기반 데이터 분할(6:2:2) 복구 중...")
times = pl.read_parquet(ADVANCED_PATH, columns=["time_group"]).sort("time_group")
train_cutoff = times["time_group"][int(len(times) * 0.6)]
val_cutoff = times["time_group"][int(len(times) * 0.8)]
del times; gc.collect()

# 3. 기본 피처 리스트 추출
temp_lf = pl.scan_parquet(ADVANCED_PATH)
exclude = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "ts", "date"]
base_numeric_features = [n for n, t in temp_lf.collect_schema().items() if n not in exclude and t.is_numeric()]

# 4. 데이터 로딩 함수 (Pure-8 + ADAN-X 16 + Base) [cite: 103-104]
def get_16dim_data(start_time, end_time):
    feat_lazy = pl.scan_parquet(ADVANCED_PATH)
    pure_lazy = pl.scan_parquet(PURE_ADAN_PATH)
    x_lazy = pl.scan_parquet("/home/tracerofjageum/adan_x_embs_dim16.parquet")
    
    if end_time: # Validation Set
        feat_lazy = feat_lazy.filter((pl.col("time_group") >= start_time) & (pl.col("time_group") < end_time))
    elif start_time == train_cutoff: # Train Set
        feat_lazy = feat_lazy.filter(pl.col("time_group") < start_time)
    else: # Test Set
        feat_lazy = feat_lazy.filter(pl.col("time_group") >= start_time)
        
    final = (feat_lazy.join(pure_lazy, on="account_id", how="left")
             .join(x_lazy, on="account_id", how="left")
             .with_columns(pl.from_epoch("time_group", time_unit="s").dt.date().alias("date"))
             .fill_null(0.0).collect())
    
    pure_features = [f"adan_emb_{i}" for i in range(8)]
    x_features = [f"emb_{i}" for i in range(64 + 16)]
    current_features = base_numeric_features + pure_features + x_features
    return final, current_features

# 5. 메인 분석 프로세스 실행
print("🚀 ADAN-X 16-dim 통합 분석 파이프라인 가동...")
df_train, feat_names_16 = get_16dim_data(train_cutoff, None)
df_val, _ = get_16dim_data(train_cutoff, val_cutoff)
df_test, _ = get_16dim_data(val_cutoff, None)

X_train, y_train = df_train.select(feat_names_16).to_numpy(), df_train["is_laundering"].to_numpy()
X_val, y_val = df_val.select(feat_names_16).to_numpy(), df_val["is_laundering"].to_numpy()
X_test, y_test = df_test.select(feat_names_16).to_numpy(), df_test["is_laundering"].to_numpy()

# 피처 다이어트 (Elite 29개 선별)
diet_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, tree_method="hist", device="cuda")
diet_model.fit(X_train, y_train)
keep_idx_16 = [i for i, imp in enumerate(diet_model.feature_importances_) if imp >= 0.001]
elite_feats_16 = [feat_names_16[i] for i in keep_idx_16]

# 최종 모델 학습 (ADAN-8 최적 파라미터 적용)
clf_16 = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, 
                           tree_method="hist", device="cuda", scale_pos_weight=48)
clf_16.fit(X_train[:, keep_idx_16], y_train, eval_set=[(X_val[:, keep_idx_16], y_val)], verbose=False)

# 6. 결과 리포트 (Daily Top-30) 
y_prob = clf_16.predict_proba(X_test[:, keep_idx_16])[:, 1]
report_df = df_test.with_columns([pl.Series("prob", y_prob)])

daily_results = []
for d in report_df["date"].unique().sort():
    day_df = report_df.filter(pl.col("date") == d)
    day_distinct = day_df.sort("prob", descending=True).unique(subset=["account_id"], maintain_order=True)
    top30_hits = day_distinct.head(30)["is_laundering"].sum()
    daily_results.append({"Date": d, "Actual_Fraud": int(day_distinct["is_laundering"].sum()), "Top-30_Hits": int(top30_hits)})

# 7. 피처 중요도 시각화
fi_df = pd.DataFrame({'feature': elite_feats_16, 'importance': clf_16.feature_importances_}).sort_values('importance', ascending=False)
fi_df['Category'] = fi_df['feature'].apply(lambda x: 'Pure-Behavior' if 'adan_emb' in x else ('Hybrid-X' if 'emb_' in x else 'Traditional'))

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(12), x='importance', y='feature', hue='Category', dodge=False)
plt.title(f'Top Killer Features (ADAN-X 16D)')
plt.show()

# 최종 출력
y_pred = (y_prob >= np.max(y_prob)*0.8).astype(int)
print("\n" + "💎"*30)
print(f"📊 ADAN-Ultimate 16-dim 복구 결과 보고서")
print("-" * 60)
print(pd.DataFrame(daily_results).to_string(index=False))
print("-" * 60)
print(f"📢 F1-Score: {f1_score(y_test, y_pred):.4f} | AUPRC: {average_precision_score(y_test, y_prob):.4f}")
print("💎"*30)

📏 시간 기반 데이터 분할(6:2:2) 복구 중...
🚀 ADAN-X 16-dim 통합 분석 파이프라인 가동...


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [07:51:04] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


ValueError: year 94647 is out of range